In [9]:
# read sms spam csv 

import pandas as pd

df = pd.read_csv(
    "SMSSpamCollection",
    sep='\t',
    header=None,
    names=['label', 'message']
)

print(df.head())






  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [10]:
print(df.head())
print(df.info())


  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB
None


In [6]:
print(df.isnull().sum())

label      0
message    0
dtype: int64


In [11]:
df['label'] = df['label'].map({
    'ham': 0,
    'spam': 1
})

print(df)

      label                                            message
0         0  Go until jurong point, crazy.. Available only ...
1         0                      Ok lar... Joking wif u oni...
2         1  Free entry in 2 a wkly comp to win FA Cup fina...
3         0  U dun say so early hor... U c already then say...
4         0  Nah I don't think he goes to usf, he lives aro...
...     ...                                                ...
5567      1  This is the 2nd time we have tried 2 contact u...
5568      0               Will ü b going to esplanade fr home?
5569      0  Pity, * was in mood for that. So...any other s...
5570      0  The guy did some bitching but I acted like i'd...
5571      0                         Rofl. Its true to its name

[5572 rows x 2 columns]


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

X = df['message']
y = df['label']

vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(X)

print(type(X))


<class 'scipy.sparse.csr.csr_matrix'>


In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [14]:
print(X_train.shape)
print(X_test.shape)

(4457, 8713)
(1115, 8713)


In [17]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
model.fit(X_train, y_train)


MultinomialNB()

In [18]:
model.fit(X_train, y_train)

MultinomialNB()

In [23]:

model.fit(X_train, y_train)
probs = model.predict_proba(X_test)
threshold = 0.3
predictions = (probs[:, 1] >= threshold).astype(int)

print(predictions)

[0 0 0 ... 0 0 0]


In [26]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, predictions)
print(accuracy)

0.9775784753363229


In [27]:
from sklearn.metrics import classification_report

print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       966
           1       0.97      0.86      0.91       149

    accuracy                           0.98      1115
   macro avg       0.97      0.93      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [31]:
from sklearn.linear_model import LogisticRegression
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
probs_lr = log_model.predict_proba(X_test)

threshold = 0.3
predictions_lr = (probs_lr[:, 1] >= threshold).astype(int)


In [33]:
from sklearn.metrics import classification_report

print("Logistic Regression Results:")
print(classification_report(y_test, predictions_lr))

Logistic Regression Results:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       966
           1       0.95      0.90      0.92       149

    accuracy                           0.98      1115
   macro avg       0.97      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [35]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, predictions_lr))

Accuracy: 0.9802690582959641


In [36]:
final_model = LogisticRegression(max_iter=1000)
final_model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [37]:
import joblib

joblib.dump(final_model, "spam_model.pkl")

['spam_model.pkl']

In [61]:
sample = ["You have received a payment of $5,000. Verify your account to claim it"]

vector = vectorizer.transform(sample)

proba = final_model.predict_proba(vector)

prediction = (proba[:,1] >= 0.3).astype(int)

print(prediction)  

[1]


In [64]:
text = ["Win a brand new smartphone today! Click the link to enter the giveaway"]
vector = vectorizer.transform(text)
pred = log_model.predict(vector)
if pred[0] == 1:
    print("Spam")
else:
    print("Ham")

Ham
